In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

project_root = os.path.abspath("..")
data_path = os.path.join(project_root, "data", "processed")
output_path = os.path.join(project_root, "Advance_Analytics")

os.makedirs(output_path, exist_ok=True)


In [2]:
returns = pd.read_csv(
    os.path.join(data_path, "cleaned_nav_history.csv")
)

returns['pct_change'] = returns.groupby('amfi_code')['nav'].pct_change()


In [3]:
# TASK 1: VaR & CVaR
grouped = returns.dropna().groupby('amfi_code')['pct_change']

var_95 = grouped.quantile(0.05)
cvar_95 = grouped.apply(lambda x: x[x <= x.quantile(0.05)].mean())

report = pd.DataFrame({
    'amfi_code': var_95.index,
    'VaR_95': var_95.values,
    'CVaR_95': cvar_95.values
})

report.to_csv(
    os.path.join(output_path, "var_cvar_report.csv"),
    index=False
)

print(report.head())
print("Saved var_cvar_report.csv")


   amfi_code    VaR_95   CVaR_95
0     100016 -0.012884 -0.016768
1     100025 -0.003338 -0.004581
2     100033 -0.016902 -0.021850
3     101206 -0.012173 -0.016075
4     101207 -0.023915 -0.030289
Saved var_cvar_report.csv


In [4]:
# TASK 2: Rolling 90-Day Sharpe

returns['date_id'] = pd.to_datetime(returns['date_id'])

pivot_df = returns.pivot_table(
    index='date_id',
    columns='amfi_code',
    values='pct_change'
)

rolling_mean = pivot_df.rolling(90).mean()
rolling_std = pivot_df.rolling(90).std()

sharpe_df = (rolling_mean / rolling_std) * np.sqrt(252)

plt.figure(figsize=(12,6))
sharpe_df.iloc[:, :5].plot(ax=plt.gca())
plt.title("Rolling 90-Day Sharpe Ratio")
plt.xlabel("Date")
plt.ylabel("Sharpe Ratio")
plt.tight_layout()

plt.savefig(
    os.path.join(output_path, "rolling_sharpe_chart.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved rolling_sharpe_chart.png")


Saved rolling_sharpe_chart.png


## Advanced Insights

1. Funds with the most negative VaR and CVaR exhibit the highest downside risk.
2. Rolling Sharpe ratios reveal periods of sustained risk-adjusted outperformance.
3. Highly concentrated portfolios tend to show larger drawdowns.
4. Consistent SIP investors demonstrate better continuity and lower churn risk.
5. High fund scores are generally associated with strong Sharpe and Alpha values.


In [5]:

import pandas as pd

transactions = pd.read_csv(
    "../data/processed/cleaned_investor_transactions.csv"
)

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

In [6]:
# First transaction year
first_txn = (
    transactions.groupby("investor_id")["transaction_date"]
    .min()
    .reset_index()
)

first_txn["cohort_year"] = (
    first_txn["transaction_date"].dt.year
)

transactions = transactions.merge(
    first_txn[["investor_id", "cohort_year"]],
    on="investor_id",
    how="left"
)


In [7]:
# Cohort metrics
cohort_analysis = (
    transactions.groupby("cohort_year")
    .agg(
        avg_sip_amount=("amount_inr", "mean"),
        total_invested=("amount_inr", "sum")
    )
    .reset_index()
)

In [8]:
# Top AMFI code per cohort
top_fund = (
    transactions.groupby(
        ["cohort_year", "amfi_code"]
    )
    .size()
    .reset_index(name="txn_count")
)

top_fund = top_fund.loc[
    top_fund.groupby("cohort_year")[
        "txn_count"
    ].idxmax()
]

cohort_analysis = cohort_analysis.merge(
    top_fund[["cohort_year", "amfi_code"]],
    on="cohort_year"
)

cohort_analysis.rename(
    columns={"amfi_code": "top_fund_amfi"},
    inplace=True
)

cohort_analysis.to_csv(
    "cohort_analysis.csv",
    index=False
)

print(cohort_analysis)

   cohort_year  avg_sip_amount  total_invested  top_fund_amfi
0         2024   107422.541832      3491125187         148568
1         2025   109158.577061        30455243         119599


In [11]:
# ==========================================
# SIP CONTINUITY ANALYSIS
# ==========================================

sip_df = transactions[
    transactions["transaction_type"] == "SIP"
].copy()

# Investors with at least 6 SIP transactions
sip_counts = (
    sip_df.groupby("investor_id")
    .size()
)

eligible_investors = sip_counts[
    sip_counts >= 6
].index

sip_df = sip_df[
    sip_df["investor_id"].isin(
        eligible_investors
    )
]

# Sort transactions chronologically
sip_df = sip_df.sort_values(
    ["investor_id", "transaction_date"]
)

# Calculate gap between SIP dates
sip_df["gap_days"] = (
    sip_df.groupby("investor_id")[
        "transaction_date"
    ]
    .diff()
    .dt.days
)

# Average gap per investor
continuity = (
    sip_df.groupby("investor_id")
    .agg(
        avg_gap_days=("gap_days", "mean")
    )
    .reset_index()
)

# Flag at-risk investors
continuity["status"] = continuity[
    "avg_gap_days"
].apply(
    lambda x: "At Risk"
    if x > 35
    else "Active"
)

# Save report
continuity.to_csv(
    "sip_continuity_report.csv",
    index=False
)

# Summary metrics
total_investors = len(continuity)

at_risk_investors = len(
    continuity[
        continuity["status"] == "At Risk"
    ]
)

active_investors = (
    total_investors
    - at_risk_investors
)

continuity_rate = (
    active_investors
    / total_investors
) * 100

print("=" * 50)
print("SIP CONTINUITY ANALYSIS")
print("=" * 50)

print(f"Eligible Investors : {total_investors}")
print(f"Active Investors   : {active_investors}")
print(f"At-Risk Investors  : {at_risk_investors}")
print(f"Continuity Rate    : {continuity_rate:.2f}%")

print("\nSample Report:")
print(continuity.head())

SIP CONTINUITY ANALYSIS
Eligible Investors : 1362
Active Investors   : 30
At-Risk Investors  : 1332
Continuity Rate    : 2.20%

Sample Report:
  investor_id  avg_gap_days   status
0   INV000004     85.400000  At Risk
1   INV000008     70.400000  At Risk
2   INV000010     64.800000  At Risk
3   INV000011     40.166667  At Risk
4   INV000012     57.000000  At Risk


In [ ]:
# ==========================================
# SECTOR HHI CONCENTRATION ANALYSIS
# ==========================================

portfolio = pd.read_csv(
    "../data/processed/cleaned_portfolio_holdings.csv"
)

# Convert percentage to decimal
portfolio["weight"] = (
    portfolio["weight_pct"] / 100
)

# Aggregate weights by fund and sector
sector_weights = (
    portfolio.groupby(
        ["amfi_code", "sector"]
    )["weight"]
    .sum()
    .reset_index()
)

# HHI Calculation
hhi = (
    sector_weights.groupby("amfi_code")["weight"]
    .apply(lambda x: (x ** 2).sum())
    .reset_index()
)

hhi.columns = [
    "amfi_code",
    "HHI"
]

# Sort by concentration
hhi = hhi.sort_values(
    "HHI",
    ascending=False
)

# Save output
hhi.to_csv(
    "sector_hhi.csv",
    index=False
)

print("=" * 50)
print("SECTOR HHI ANALYSIS")
print("=" * 50)

print("\nMost Concentrated Funds:")
print(hhi.head())

print("\nMost Diversified Funds:")
print(hhi.tail())

print("\nSaved: sector_hhi.csv")

SECTOR HHI ANALYSIS

Most Concentrated Funds:
    amfi_code       HHI
11     119092  0.296769
30     148569  0.254992
27     125498  0.253155
6      102887  0.251383
32     149323  0.241077

Most Diversified Funds:
    amfi_code       HHI
9      118634  0.160299
14     119095  0.159582
15     119551  0.142491
25     120843  0.136206
5      102886  0.124020

Saved: sector_hhi.csv


: 